# Gemma 4 AML Fine-Tune -- GGUF -- HuggingFace Hub

**Pipeline:**
1. Install Unsloth + deps
2. QLoRA fine-tune on `framl_train_combined_v26.jsonl` (713 examples)
3. Merge LoRA -> export F16 GGUF -> quantise Q4_K_M
4. Upload GGUF to HuggingFace Hub
5. Register with Ollama as `aria-v2`

> **NOTE:** Gemma 4 model IDs and chat template tokens should be verified
> against the Unsloth release notes before running -- marked with `# VERIFY` comments.


## Cell 1 -- Install Dependencies
Unsloth must be installed before transformers/trl.

In [1]:
!/venv/main/bin/pip install unsloth unsloth_zoo transformers trl peft datasets \
    bitsandbytes accelerate sentence-transformers huggingface_hub

# torchcodec pulls in FFmpeg 4.x dependency that conflicts with vast.ai system FFmpeg 5+
!/venv/main/bin/pip uninstall -y torchcodec 2>/dev/null || true

# Flash Attention 2 -- try precompiled wheel only (--only-binary=:all: makes pip fail
# instantly if no wheel exists rather than hanging on a 20-min source build).
# Unsloth has its own built-in attention kernels so training works fine without it.
import subprocess
fa_result = subprocess.run(
    ["/venv/main/bin/pip", "install", "flash-attn",
     "--only-binary=:all:", "--no-deps", "--quiet"],
    capture_output=True, text=True
)
if fa_result.returncode == 0:
    print("flash-attn: installed")
else:
    print("flash-attn: no precompiled wheel for this torch/CUDA version -- skipping")
    print("  Unsloth built-in kernels are active, training unaffected.")

import unsloth; print("Unsloth:", unsloth.__version__)

INFO: pip is looking at multiple versions of torchvision to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.0/67.0 MB 65.5 MB/s  0:00:01m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 63.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 67.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.6/915.6 MB 61.0 MB/s  0:00:13m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 64.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 62.5 MB/s  0:00:09m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 65.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 65.1 MB/s  0:00:01 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 954.8/954.8 kB 48.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.1/193.1 MB 65.5 MB/s  0:00:02m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━

## Cell 2 -- Environment & Imports
`HF_HOME` must be set **before** any HuggingFace import.

In [ ]:
import os
os.environ['HF_HOME'] = '/workspace/huggingface'   # RAM-backed tmpfs -- fast, ~15GB free, wiped on stop
os.environ['HF_TOKEN'] = 'hf_PASTE_YOUR_TOKEN_HERE'  # paste your token here -- NEVER commit a real token

import torch
from pathlib import Path
from unsloth import FastLanguageModel
from trl import SFTConfig, SFTTrainer
from datasets import Dataset

print(f'CUDA: {torch.cuda.is_available()} | GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none"}')
if torch.cuda.is_available():
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## Cell 3 -- Configuration

**GPU guide:**
- RTX 3090 / 4090 (24 GB) -> `gemma-4-4b-it` (E4B), batch=2, accum=4
- A100 40 GB -> `gemma-4-27b-it`, batch=1, accum=8


In [ ]:
# -- Model --
BASE_MODEL  = 'unsloth/gemma-4-E4B-it'   # VERIFY: check Unsloth hub for exact ID
MAX_SEQ_LEN = 4096

# -- LoRA --
LORA_R       = 16
LORA_ALPHA   = 32
LORA_DROPOUT = 0.05
TARGET_MODULES = ['q_proj', 'k_proj', 'v_proj', 'o_proj',
                  'gate_proj', 'up_proj', 'down_proj']

# -- Training --
EPOCHS     = 3        # 1017 examples x 3 epochs
BATCH_SIZE = 2
GRAD_ACCUM = 4        # effective batch = 8
LR         = 2e-4

# -- Paths --
REPO_ROOT = Path.cwd()
if REPO_ROOT.name == 'finetune':
    REPO_ROOT = REPO_ROOT.parent
DATA_FILE  = REPO_ROOT / 'finetune' / 'data' / 'aria_train_combined_v42_full.jsonl'
OUT_DIR    = Path('/workspace/aria-v6-adapter')
GGUF_DIR   = Path('/workspace/aria-v6-gguf')
F16_GGUF   = Path('/workspace/aria-v6-f16.gguf')   # intermediate -- deleted after both quants
Q8_GGUF    = Path('/workspace/aria-v6-q8.gguf')    # ~8 GB
Q4KM_GGUF  = Path('/workspace/aria-v6-q4km.gguf')  # ~2.5 GB
OUT_DIR.mkdir(parents=True, exist_ok=True)
GGUF_DIR.mkdir(parents=True, exist_ok=True)

# -- HuggingFace Hub --
HF_REPO  = 'speri420/aria-v6'
HF_TOKEN = os.environ.get('HF_TOKEN', '')

# -- Ollama --
OLLAMA_MODEL_NAME = 'aria-v6'

print(f'Base model: {BASE_MODEL}')
print(f'Data:       {DATA_FILE}  (exists={DATA_FILE.exists()})')
print(f'GGUF out:   {Q4KM_GGUF}  (Q8: {Q8_GGUF.name})')
print(f'HF repo:    {HF_REPO}  (token set={bool(HF_TOKEN)})')

## Cell 4 -- Load Base Model (4-bit QLoRA)

In [4]:
print(f'Loading {BASE_MODEL} in 4-bit QLoRA ...')
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = BASE_MODEL,
    max_seq_length = MAX_SEQ_LEN,
    dtype          = None,
    load_in_4bit   = True,
)
print('Model loaded.')
tmpl = tokenizer.chat_template or 'NOT SET'
print(f'Chat template (first 120): {tmpl[:120]}')


Loading unsloth/gemma-4-E4B-it in 4-bit QLoRA ...
==((====))==  Unsloth 2026.4.8: Fast Gemma4 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA GeForce RTX 3090 Ti. Num GPUs = 1. Max memory: 23.53 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.6. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/2130 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/208 [00:00<?, ?B/s]

processor_config.json: 0.00B [00:00, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/32.2M [00:00<?, ?B/s]

Model loaded.
Chat template (first 120): {%- macro format_parameters(properties, required) -%}
    {%- set standard_keys = ['description', 'type', 'properties', 


## Cell 5 -- Attach LoRA Adapters

In [5]:
model = FastLanguageModel.get_peft_model(
    model,
    r              = LORA_R,
    lora_alpha     = LORA_ALPHA,
    lora_dropout   = LORA_DROPOUT,
    target_modules = TARGET_MODULES,
    bias           = 'none',
    use_gradient_checkpointing = 'unsloth',
    random_state   = 42,
)
model.print_trainable_parameters()


Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
[unsloth_zoo.log|WARNING]Unsloth: Failed to register input-embedding hook for `model.base_model.model.model.audio_tower`: `get_input_embeddings` not auto‑handled for Gemma4AudioModel; please override in the subclass.. Falling back to pre-forward hook.


trainable params: 42,401,792 || all params: 8,038,558,240 || trainable%: 0.5275


## Cell 6 -- Load & Format Dataset

Gemma 4 uses `<start_of_turn>` / `<end_of_turn>` tokens.
`tokenizer.apply_chat_template()` handles the format automatically.

> **VERIFY:** If tool_calls turns cause errors, the Unsloth Gemma 4 tokenizer
> may need tool role handling -- check Unsloth release notes.


In [6]:
import json

def load_jsonl(path):
    records = []
    with open(path, encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records

def format_example(example):
    msgs = example['messages']
    # Pass messages through unchanged -- apply_chat_template handles tool_calls
    # and role:tool natively, producing the correct Gemma 4 format:
    #   <|tool_call>call:name{"arg": "val"}<tool_call|>   (tool call)
    #   <|turn>tool{content}<turn|>                      (tool result)
    text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)
    return {'text': text}

raw = load_jsonl(str(DATA_FILE))
print(f'Loaded {len(raw)} examples from {DATA_FILE.name}')
dataset = Dataset.from_list(raw).map(format_example, remove_columns=['messages'])
print(f'Formatted: {len(dataset)} rows')
print('Sample (first 800 chars):')
print(dataset[0]['text'][:800])


Loaded 956 examples from aria_train_combined_v39_full.jsonl


Map:   0%|          | 0/956 [00:00<?, ? examples/s]

Formatted: 956 rows
Sample (first 800 chars):
<bos><|turn>system
You are ARIA — Agentic Risk Intelligence for AML. You identify natural customer behavioral segments using unsupervised K-Means clustering and explain their AML risk profiles. You analyze false positive/false negative trade-offs in AML alert thresholds, perform customer behavioral segmentation, and interpret clustering results. Use the available tools to retrieve data, then provide clear, analytical insights. Be concise and reference specific numbers when interpreting results.<turn|>
<|turn>user
Cluster individual customers by transaction behavior into 4 groups<turn|>
<|turn>model
<|tool_call>call:cluster_analysis{{"customer_type": "Individual", "n_clusters": 4}}<tool_call|><turn|>
<|turn>tool
Tool result for cluster_analysis:
Clustering complete. customer_type=Individual


## Cell 7 -- Train

In [7]:
sft_config = SFTConfig(
    output_dir                  = str(OUT_DIR),
    num_train_epochs            = EPOCHS,
    per_device_train_batch_size = BATCH_SIZE,
    gradient_accumulation_steps = GRAD_ACCUM,
    learning_rate               = LR,
    lr_scheduler_type           = 'cosine',
    warmup_ratio                = 0.05,
    fp16                        = not torch.cuda.is_bf16_supported(),
    bf16                        = torch.cuda.is_bf16_supported(),
    logging_steps               = 10,
    save_strategy               = 'epoch',
    optim                       = 'adamw_8bit',
    dataset_text_field          = 'text',
    max_seq_length              = MAX_SEQ_LEN,
    dataset_num_proc            = 2,
    report_to                   = 'none',
)
trainer = SFTTrainer(
    model         = model,
    tokenizer     = tokenizer,
    train_dataset = dataset,
    args          = sft_config,
)
print(f'Training: {len(dataset)} examples | {EPOCHS} epochs | effective batch={BATCH_SIZE * GRAD_ACCUM}')
trainer.train()
print('Training complete.')
trainer.save_model(str(OUT_DIR))
tokenizer.save_pretrained(str(OUT_DIR))
print(f'Adapter saved to {OUT_DIR}')


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/956 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': 2}.


Training: 956 examples | 3 epochs | effective batch=8


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 956 | Num Epochs = 3 | Total steps = 360
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 42,401,792 of 8,038,558,240 (0.53% trained)
Caching is incompatible with gradient checkpointing in Gemma4TextDecoderLayer. Setting `past_key_values=None`.


...

Step,Training Loss
10,8.643410
20,4.182714
30,2.238007
40,1.078131
50,0.663275
60,0.486108
70,0.391531
80,0.356539
90,0.281885
100,0.250067


Unsloth: Restored added_tokens_decoder metadata in /workspace/aria-v2-adapter/checkpoint-120/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /workspace/aria-v2-adapter/checkpoint-240/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /workspace/aria-v2-adapter/checkpoint-360/tokenizer_config.json.


Training complete.


Unsloth: Restored added_tokens_decoder metadata in /workspace/aria-v2-adapter/tokenizer_config.json.


Adapter saved to /workspace/aria-v2-adapter


## Cell 8a -- Build llama.cpp (skip if already built)

In [8]:
import subprocess, shutil, sys

CUDA_ARCH = '89'  # RTX 3090; A100=80; H100=90

# Optional: pin llama.cpp to a specific commit to avoid regressions in
# convert_hf_to_gguf.py for Gemma 4.  Leave blank to use latest HEAD.
# To find the commit used in a prior working build:
#   cd /workspace/llama.cpp && git rev-parse HEAD
# Paste the result here, e.g.: LLAMA_CPP_COMMIT = "abc1234"
LLAMA_CPP_COMMIT = ""  # e.g. "abc1234" -- blank = latest HEAD

llama_q = Path('/workspace/llama.cpp/build/bin/llama-quantize')
if llama_q.exists():
    # Print the commit so we know which version produced the GGUF
    rev = subprocess.run('cd /workspace/llama.cpp && git rev-parse --short HEAD',
                         shell=True, capture_output=True, text=True)
    print(f'llama.cpp already built -- skipping.  commit={rev.stdout.strip()}')
else:
    cmds = [
        'apt-get install -y cmake build-essential 2>/dev/null',
        'cd /workspace && git clone https://github.com/ggerganov/llama.cpp || true',
    ]
    if LLAMA_CPP_COMMIT:
        cmds.append(f'cd /workspace/llama.cpp && git checkout {LLAMA_CPP_COMMIT}')
    cmds.append(
        f'cd /workspace/llama.cpp && cmake -B build -DGGML_CUDA=ON -DCMAKE_CUDA_ARCHITECTURES={CUDA_ARCH} && cmake --build build --config Release -j$(nproc)'
    )
    for cmd in cmds:
        print(f'>> {cmd[:80]}')
        r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
        if r.returncode != 0:
            print('STDERR:', r.stderr[-500:])
            raise RuntimeError(f'Failed: {cmd}')
    rev = subprocess.run('cd /workspace/llama.cpp && git rev-parse --short HEAD',
                         shell=True, capture_output=True, text=True)
    print(f'llama.cpp build complete.  commit={rev.stdout.strip()}')

LLAMA_CONVERT  = Path('/workspace/llama.cpp/convert_hf_to_gguf.py')
LLAMA_QUANTIZE = Path('/workspace/llama.cpp/build/bin/llama-quantize')
print(f'convert:  {LLAMA_CONVERT.exists()}')
print(f'quantize: {LLAMA_QUANTIZE.exists()}')

llama.cpp already built -- skipping.  commit=d8794eecd
convert:  True
quantize: True


## Cell 8b -- Merge LoRA -> F16 GGUF -> Q4_K_M GGUF

In [ ]:
import os, subprocess, shutil as _shutil, sys
from pathlib import Path

# Set to True when retraining to force a clean merge + conversion
FORCE_RETRAIN = True

if FORCE_RETRAIN:
    for _p in [GGUF_DIR, F16_GGUF, Q8_GGUF, Q4KM_GGUF]:
        _p = Path(_p)
        if _p.is_dir():
            print(f'Removing stale dir {_p}')
            _shutil.rmtree(str(_p))
        elif _p.exists():
            print(f'Removing stale file {_p}')
            _p.unlink()

# -- Step 1: Merge LoRA -> float16 safetensors --------------------------------
if not list(GGUF_DIR.glob('*.safetensors')):
    print('Merging LoRA -> float16 safetensors ...')
    if GGUF_DIR.exists(): _shutil.rmtree(str(GGUF_DIR))
    GGUF_DIR.mkdir(parents=True, exist_ok=True)
    model.save_pretrained_merged(str(GGUF_DIR), tokenizer, save_method="merged_16bit")
    safetensors = list(GGUF_DIR.glob('*.safetensors'))
    total_gb = sum(f.stat().st_size for f in safetensors) / 1e9
    print(f'Merged -> {GGUF_DIR}  ({len(safetensors)} shards, {total_gb:.1f} GB)')
else:
    print('Merged weights exist -- skipping merge.')

# -- Step 2: Convert safetensors -> f16 GGUF (~16 GB) -------------------------
# f16 is the required source for both q8_0 and Q4_K_M quantization.
# Safetensors are deleted here to free disk; f16 GGUF stays until both quants done.
if not F16_GGUF.exists():
    print('Converting to f16 GGUF ...')
    r = subprocess.run(
        [sys.executable, str(LLAMA_CONVERT), str(GGUF_DIR),
         '--outfile', str(F16_GGUF), '--outtype', 'f16'],
        capture_output=True, text=True
    )
    if r.returncode != 0:
        print(r.stderr[-1500:]); raise RuntimeError('f16 conversion failed')
    f16_gb = F16_GGUF.stat().st_size / 1e9
    print(f'f16 GGUF: {f16_gb:.1f} GB')
    assert f16_gb > 5.0, f'f16 GGUF suspiciously small ({f16_gb:.2f} GB) -- aborting'
    print('Removing merged safetensors to free disk ...')
    _shutil.rmtree(str(GGUF_DIR))
    subprocess.run('df -h /workspace', shell=True)
else:
    print(f'f16 GGUF exists ({F16_GGUF.stat().st_size / 1e9:.1f} GB) -- skipping.')

In [ ]:
if not F16_GGUF.exists():
    raise FileNotFoundError(f'{F16_GGUF} not found -- run Step 2 first')

# -- Step 3: Quantise f16 -> q8_0 (~8 GB) ------------------------------------
if not Q8_GGUF.exists():
    print(f'Quantising f16 -> q8_0 ...')
    r = subprocess.run(
        [str(LLAMA_QUANTIZE), str(F16_GGUF), str(Q8_GGUF), 'Q8_0'],
        capture_output=True, text=True
    )
    if r.returncode != 0:
        print(r.stderr[-1500:]); raise RuntimeError('q8_0 quantisation failed')
    print(f'q8_0: {Q8_GGUF.stat().st_size / 1e9:.1f} GB')
else:
    print(f'q8_0 exists ({Q8_GGUF.stat().st_size / 1e9:.1f} GB) -- skipping.')

# -- Step 4: Quantise f16 -> Q4_K_M (~2.5 GB) ---------------------------------
if not Q4KM_GGUF.exists():
    print(f'Quantising f16 -> Q4_K_M ...')
    r = subprocess.run(
        [str(LLAMA_QUANTIZE), str(F16_GGUF), str(Q4KM_GGUF), 'Q4_K_M'],
        capture_output=True, text=True
    )
    if r.returncode != 0:
        print(r.stderr[-1500:]); raise RuntimeError('Q4_K_M quantisation failed')
    q4_gb = Q4KM_GGUF.stat().st_size / 1e9
    print(f'Q4_K_M: {q4_gb:.1f} GB')
    assert q4_gb > 1.0, f'Q4_K_M suspiciously small ({q4_gb:.2f} GB) -- check output'
else:
    print(f'Q4_K_M exists ({Q4KM_GGUF.stat().st_size / 1e9:.1f} GB) -- skipping.')

# Delete f16 only after both quants are confirmed on disk
if Q8_GGUF.exists() and Q4KM_GGUF.exists():
    print('Both quants confirmed -- removing f16 GGUF to free disk ...')
    F16_GGUF.unlink()
    subprocess.run('df -h /workspace', shell=True)
else:
    print('WARNING: one or both quants missing -- keeping f16 GGUF')

## Cell 8c -- Merge + q8_0 GGUF + Q4_K_M (disk-space-aware pipeline)

Use this instead of 8b when disk is tight (&lt;20GB free).
Converts to q8_0 (~8GB) first, deletes the 15GB safetensors, then quantises to Q4_K_M.

In [11]:
from huggingface_hub import HfApi
from pathlib import Path

# Upload whichever GGUF exists -- q8_0 if Q4_K_M not yet produced
Q8_GGUF  = Path('/workspace/aria-v2-q8.gguf')
UPLOAD_FILE = Q4KM_GGUF if Q4KM_GGUF.exists() else Q8_GGUF
if not UPLOAD_FILE.exists():
    raise FileNotFoundError('No GGUF file found -- run Cell 8b or 8c first.')

api = HfApi(token=HF_TOKEN)
try:
    api.create_repo(repo_id=HF_REPO, repo_type='model', exist_ok=True)
    print(f'Repo ready: {HF_REPO}')
except Exception as e:
    print(f'Repo: {e}')

print(f'Uploading {UPLOAD_FILE.name} ({UPLOAD_FILE.stat().st_size / 1e9:.1f} GB) ...')
api.upload_file(
    path_or_fileobj = str(UPLOAD_FILE),
    path_in_repo    = UPLOAD_FILE.name,
    repo_id         = HF_REPO,
    repo_type       = 'model',
)
print(f'Done: https://huggingface.co/{HF_REPO}')
print(f'File uploaded: {UPLOAD_FILE.name}')


Repo ready: speri420/aria-v2
Uploading aria-v2-q8.gguf (8.0 GB) ...


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Done: https://huggingface.co/speri420/aria-v2
File uploaded: aria-v2-q8.gguf


## Cell 9 -- Upload GGUF to HuggingFace Hub

In [ ]:
from huggingface_hub import HfApi

api = HfApi(token=HF_TOKEN)
try:
    api.create_repo(repo_id=HF_REPO, repo_type='model', exist_ok=True)
    print(f'Repo ready: {HF_REPO}')
except Exception as e:
    print(f'Repo: {e}')

print(f'Uploading {Q4KM_GGUF.name} ({Q4KM_GGUF.stat().st_size / 1e9:.1f} GB) ...')
api.upload_file(
    path_or_fileobj = str(Q4KM_GGUF),
    path_in_repo    = Q4KM_GGUF.name,
    repo_id         = HF_REPO,
    repo_type       = 'model',
)
print(f'Done: https://huggingface.co/{HF_REPO}')


## Cell 10 -- Install Ollama (skip if already installed)

In [12]:
import subprocess, shutil

if shutil.which("ollama"):
    print("Ollama already installed:", shutil.which("ollama"))
else:
    print("Installing Ollama ...")
    r = subprocess.run(
        "curl -fsSL https://ollama.com/install.sh | sh",
        shell=True, capture_output=True, text=True
    )
    print(r.stdout[-500:])
    if r.returncode != 0:
        print(r.stderr[-500:])
        raise RuntimeError("Ollama install failed")
    print("Ollama installed.")

r = subprocess.run(["ollama", "--version"], capture_output=True, text=True)
print(r.stdout.strip())


Installing Ollama ...

Ollama installed.


## Cell 11 -- Write Modelfile & Register with Ollama

Gemma 4 uses `<start_of_turn>` / `<end_of_turn>` chat tokens.

> **VERIFY:** Confirm the TEMPLATE syntax against the Ollama Gemma library page
> before running: https://ollama.com/library/gemma3 (use equivalent for Gemma 4)


In [ ]:
import subprocess
from pathlib import Path

if not Q4KM_GGUF.exists():
    raise FileNotFoundError(f'{Q4KM_GGUF} not found -- run Cell 8b first.')

print(f'Using GGUF: {Q4KM_GGUF}  ({Q4KM_GGUF.stat().st_size / 1e9:.1f} GB)')

SYSTEM_PROMPT = (
    'You are ARIA, an AML (Anti-Money Laundering) analytics AI assistant built by Xceed. '
    'You analyze false positive/false negative trade-offs in AML alert thresholds, '
    'perform customer behavioral segmentation, and interpret clustering results. '
    'Use the available tools to retrieve data, then provide clear, analytical insights. '
    'IMPORTANT: You MUST respond entirely in English. '
    'Be concise and reference specific numbers when interpreting results.'
)

lines = [
    f'FROM {Q4KM_GGUF}',
    '',
    'PARAMETER num_ctx 8192',
    'PARAMETER temperature 0.0',
    'PARAMETER top_p 0.9',
    'PARAMETER stop <turn|>',
    'PARAMETER stop <eos>',
    '',
    'TEMPLATE """',
    '{{ if .System }}<|turn>user',
    '{{ .System }}<turn|>',
    '{{ end }}',
    '{{ range .Messages }}',
    '{{ if eq .Role "user" }}<|turn>user',
    '{{ .Content }}<turn|>',
    '<|turn>model',
    '{{ else if eq .Role "assistant" }}{{ .Content }}<turn|>',
    '{{ else if eq .Role "tool" }}<|turn>tool',
    '{{ .Content }}<turn|>',
    '<|turn>model',
    '{{ end }}',
    '{{ end }}',
    '"""',
    '',
    f'SYSTEM "{SYSTEM_PROMPT}"',
]

modelfile_content = chr(10).join(lines)
modelfile_path = Path('/tmp/Modelfile.aml')
modelfile_path.write_text(modelfile_content, encoding='utf-8')
print('--- Modelfile ---')
print(modelfile_content)

## Cell 12 -- Start Ollama Server

Binds to 0.0.0.0:11434. Expose via vast.ai port forwarding or Cloudflare tunnel.


In [14]:
import time

subprocess.run("pkill -f 'ollama serve' || true", shell=True)
time.sleep(2)
env = os.environ.copy()
env['OLLAMA_HOST'] = '0.0.0.0:11434'
proc = subprocess.Popen(['ollama', 'serve'], env=env,
                        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(4)
print(f'Ollama server PID: {proc.pid}')
r = subprocess.run(['curl', '-s', 'http://localhost:11434/api/tags'],
                   capture_output=True, text=True)
print('Health check:', r.stdout[:200])


Ollama server PID: 10536
Health check: {"models":[]}


In [15]:
print(f'Registering {OLLAMA_MODEL_NAME} ...')
result = subprocess.run(
    ['ollama', 'create', OLLAMA_MODEL_NAME, '-f', str(modelfile_path)],
    capture_output=True, text=True
)
print(result.stdout[-500:])
if result.returncode != 0:
    print('STDERR:', result.stderr[-500:])
    raise RuntimeError('ollama create failed')
r = subprocess.run(['ollama', 'list'], capture_output=True, text=True)
print('Registered models:')
print(r.stdout)


Registering aria-v2 ...

Registered models:
NAME              ID              SIZE      MODIFIED               
aria-v2:latest    0a7603f5b90f    8.0 GB    Less than a second ago    



## Cell 14 -- Smoke Test

In [16]:
import urllib.request, json as _json

payload = _json.dumps({
    'model': OLLAMA_MODEL_NAME,
    'messages': [{'role': 'user', 'content': 'Show FP/FN trade-off for Business customers by monthly transaction amount'}],
    'stream': False,
}).encode()
req = urllib.request.Request(
    'http://localhost:11434/api/chat',
    data=payload,
    headers={'Content-Type': 'application/json'},
)
with urllib.request.urlopen(req, timeout=120) as resp:
    result = _json.loads(resp.read())
print('Response:')
print(result['message']['content'][:800])


Response:
<mask><unused24><unused17><unused2>[multimodal]<unused28><unused22><unused18><unused6><unused4><unused13><unused12><unused8><unused15><unused20><unused11>[multimodal]<unused7><unused28><unused0><unused22><mask><unused13><unused2><unused21><unk><unused28><unused27><unused28><unused17><unused26><unused29><unused18><unused29><unused11><unused3><unused19><unused3>


In [ ]:
from unsloth import FastLanguageModel
FastLanguageModel.for_inference(model)

messages = [{"role": "user", "content": "What is your name?"}]
inputs = tokenizer.apply_chat_template(
  messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
).to("cuda")

outputs = model.generate(input_ids=inputs, max_new_tokens=100, use_cache=True)
decoded = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=False)
print(repr(decoded))


In [20]:
from unsloth import FastLanguageModel                                                                                                                                                                                             
FastLanguageModel.for_inference(model)
                                                                                                                                                                                                                                
messages = [{"role": "user", "content": "What is your name?"}]

# Step 1: format to text first
text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
print("Prompt:", repr(text[:300]))

text_tok = tokenizer.tokenizer  # underlying text tokenizer                                                                                                                                                                       

input_ids = text_tok(text, return_tensors="pt")["input_ids"].to("cuda")                                                                                                                                                           
outputs = model.generate(input_ids=input_ids, max_new_tokens=100, use_cache=True)
print(text_tok.decode(outputs[0][input_ids.shape[1]:], skip_special_tokens=False))
  


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Prompt: '<bos><|turn>user\nWhat is your name?<turn|>\n<|turn>model\n'
I am Gemma 4. I am a Large Language Model developed by Google DeepMind.<turn|>


In [22]:
#Now we need to figure out what Cell 8b produced. Run this in a new cell:                                                                                                                                                          

import subprocess                                                                                                                                                                                                                 
r = subprocess.run('ls -lh /workspace/aria-v2-q8.gguf /workspace/aria-v2-gguf/ 2>/dev/null || echo "not found"',
                 shell=True, capture_output=True, text=True)
print(r.stdout)

# Also check llama.cpp commit used
r2 = subprocess.run('cd /workspace/llama.cpp && git rev-parse --short HEAD',
                  shell=True, capture_output=True, text=True)
print("llama.cpp commit:", r2.stdout.strip())

#And also check if an mmproj file exists anywhere:

r3 = subprocess.run('find /workspace -name "*mmproj*" -o -name "*-mmproj*" 2>/dev/null',
                  shell=True, capture_output=True, text=True)
print("mmproj files:", r3.stdout)

#The [multimodal] tokens we saw in Ollama are the smoking gun — the model is trying to run its vision pathway because the text GGUF is being loaded without the mmproj file that E4B requires. If we find an mmproj file, the fix
#is registering both together in Ollama.


-rw-r--r-- 1 root root 7.5G May  4 15:14 /workspace/aria-v2-q8.gguf
not found

llama.cpp commit: d8794eecd
mmproj files: 


In [ ]:
#Let's bypass Ollama completely and test the raw GGUF with llama-cli:
                                                                                                                                                                                                                                
import subprocess

r = subprocess.run(
  ['/workspace/llama.cpp/build/bin/llama-cli',
   '-m', '/workspace/aria-v2-q8.gguf',
   '-ngl', '99',
   '--prompt', '<bos><|turn>user\nWhat is your name?<turn|>\n<|turn>model\n',
   '--temp', '0.1',
   '-n', '80',
   '--no-display-prompt'],
  capture_output=True, text=True, timeout=120
)
print("STDOUT:", r.stdout[-500:])
print("STDERR:", r.stderr[-400:])




In [2]:
import subprocess, time
                                                                                                                                                                                                                                
proc = subprocess.Popen(
  ['/workspace/llama.cpp/build/bin/llama-cli',
   '-m', '/workspace/aria-v2-q8.gguf',
   '-ngl', '99',
   '--prompt', '<bos><|turn>user\nWhat is your name?<turn|>\n<|turn>model\n',
   '--temp', '0.1', '-n', '20',
   '--no-display-prompt'],
  stdout=open('/tmp/llama_out.txt', 'w'),
  stderr=open('/tmp/llama_err.txt', 'w')
)

time.sleep(20)

with open('/tmp/llama_err.txt') as f:
  err = f.read()
with open('/tmp/llama_out.txt') as f:
  out = f.read()

print("RUNNING:", proc.poll() is None)
print("=== STDERR ===")
print(err[-800:])
print("=== STDOUT ===")
print(out)

if proc.poll() is None:
  proc.kill()


IOPub data rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_data_rate_limit`.

Current values:
ServerApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
ServerApp.rate_limit_window=3.0 (secs)



In [3]:
import subprocess                                                                                                                                                                                                                 

r = subprocess.run(                                                                                                                                                                                                               
  'tail -30 /tmp/llama_err.txt',
  shell=True, capture_output=True, text=True
)
print(r.stdout)

r2 = subprocess.run(
  'cat /tmp/llama_out.txt',
  shell=True, capture_output=True, text=True
)
print("STDOUT:", r2.stdout[:500])


ggml_cuda_init: found 1 CUDA devices (Total VRAM: 24095 MiB):
  Device 0: NVIDIA GeForce RTX 3090 Ti, compute capability 8.6, VMM: yes, VRAM: 24095 MiB

STDOUT: 
Loading model... 


▄▄ ▄▄
██ ██
██ ██  ▀▀█▄ ███▄███▄  ▀▀█▄    ▄████ ████▄ ████▄
██ ██ ▄█▀██ ██ ██ ██ ▄█▀██    ██    ██ ██ ██ ██
██ ██ ▀█▄██ ██ ██ ██ ▀█▄██ ██ ▀████ ████▀ ████▀
                                    ██    ██
                                    ▀▀    ▀▀

build      : b9022-d8794eecd
model      : aria-v2-q8.gguf
modalities : text

available commands:
  /exit or Ctrl+C     stop or exit
  /regen              regenerate the last response
  /clear         


In [ ]:
#Good news: CUDA works, model loaded clean, modalities: text — no multimodal confusion at the GGUF level. The issue is that newer llama-cli (b9022) defaults to interactive mode and ignored our --prompt flag.

#Let's check what other binaries exist and use llama-run which is designed for one-shot batch inference:                                                                                                                           

import subprocess                                                                                                                                                                                                                 
                   
# See what's available
r = subprocess.run('ls /workspace/llama.cpp/build/bin/', shell=True, capture_output=True, text=True)
print(r.stdout)

#Then try llama-run:

r = subprocess.run(
  ['/workspace/llama.cpp/build/bin/llama-run',
   '-ngl', '99',
   '/workspace/aria-v2-q8.gguf',
   '<bos><|turn>user\nWhat is your name?<turn|>\n<|turn>model\n'],
  capture_output=True, text=True, timeout=60
)
print("STDOUT:", r.stdout[:500])
print("STDERR:", r.stderr[-300:])


In [ ]:
#no-cnv to force non interactive mode
import subprocess                                                                                                                                                                                                                 
              
r = subprocess.run(
  ['/workspace/llama.cpp/build/bin/llama-cli',
   '-m', '/workspace/aria-v2-q8.gguf',
   '-ngl', '99',
   '-p', '<bos><|turn>user\nWhat is your name?<turn|>\n<|turn>model\n',
   '-n', '80',
   '-no-cnv'],
  capture_output=True, text=True, timeout=60
)
print("STDOUT:", r.stdout[-500:])
print("STDERR:", r.stderr[-300:])


In [ ]:
import subprocess
                                                                                                                                                                                                                                
# Run with empty stdin (EOF) — forces interactive mode to exit after prompt
r = subprocess.run(
  ['/workspace/llama.cpp/build/bin/llama-cli',
   '-m', '/workspace/aria-v2-q8.gguf',
   '-ngl', '99',
   '-p', 'What is your name?',
   '-n', '30'],
  input=b'',
  capture_output=True,
  timeout=60
)
print("STDOUT:", r.stdout.decode()[-500:])
print("STDERR:", r.stderr.decode()[-300:])


In [7]:
'''
yes I was checking before the error the GPU was going up to 8 GB and after the error back to 0 utilization                                                                                                                        
                                                                                                             
● That confirms GPU offload IS working (8GB VRAM = Q8_0 model fully on GPU). The model loads fine, then generation hangs — that's a llama.cpp d8794eecd bug or interactive-mode deadlock, not a CUDA problem.                       
                                                                                                                                                                                                                                    
  Here's what we know:                                                                                                                                                                                                              
  - ✅ Training good (in-memory inference clean)
  - ✅ GGUF loads on GPU (8GB VRAM, text modality)                                                                                                                                                                                  ─
  - ❌ Generation hangs in llama-cli b9022 (interactive mode issues + possible generation bug)
  - ❌ Ollama 0.23.0 produces garbage (bundled llama.cpp too old for new GGUF format)

  The cleanest fix is to upgrade Ollama — newer Ollama bundles a newer llama.cpp that understands the May 4 GGUF format changes. Run this in a cell:
'''
import subprocess

# Kill existing server first
subprocess.run('pkill -f "ollama serve" || true', shell=True)

# Upgrade Ollama
r = subprocess.run(
  'curl -fsSL https://ollama.com/install.sh | sh',
  shell=True, capture_output=True, text=True
)
print(r.stdout[-500:])
print(r.stderr[-200:])

# Check new version
r2 = subprocess.run(['ollama', '--version'], capture_output=True, text=True)
print("Version:", r2.stdout.strip())



...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.

Version: Warning: could not connect to a running Ollama instance


In [8]:
import subprocess                                                                                                                                                                                                                 
                   
# Find the commit that introduced final_logit_softcapping or unused24
r = subprocess.run(
  'cd /workspace/llama.cpp && git log --oneline | head -30',
  shell=True, capture_output=True, text=True
)
print(r.stdout)

# Also search for the specific change
r2 = subprocess.run(
  'cd /workspace/llama.cpp && git log --oneline --all | grep -i "softcap\\|unused24\\|gemma.*gguf\\|gguf.*gemma" | head -15',
  shell=True, capture_output=True, text=True
)
print(r2.stdout)


d8794eecd examples: refactor diffusion generation (#22590)
36a694c96 webui : fix circular dependency between chat.service.ts and models.svelte.ts (#22625)
a4701c98f common/autoparser: fixes for newline handling / forced tool calls (#22654)
994118a18 model: move `load_hparams` and `load_tensors` to per-model definition (#22004)
c84e6d6db server: Add a simple get_datetime server tool (#22649)
fa8feaed3 webui: restore missing settings (#22666)
846262d78 docs : update speculative decoding parameters after refactor (#22397) (#22539)
6dcd824fc vulkan: delete dead GGML_VK_MAX_NODES def (#22621)
d4b0c22f9 ggml-webgpu: add layer norm ops (#22406)
e48034dfc common : determine generation prompt using longest common prefix (#22657)
048a490f7 convert :  Mistral format yarn apply_scale support (#22612)
db44417b0 convert : apply Q/K RoPE permutation in NVFP4 repack path (#22611)
d05fe1d7d fix: CUDA device PCI bus ID de-dupe OOMing (ignoring other 3 gpus entirely) (#22533)
0754b7b6f server : avoid che

In [9]:
import subprocess
                                                                                                                                                                                                                                
# When did convert_hf_to_gguf.py last change for Gemma 4 / softcap?
r = subprocess.run(
  'cd /workspace/llama.cpp && git log --oneline -- convert_hf_to_gguf.py | head -20',
  shell=True, capture_output=True, text=True
)
print(r.stdout)

# Find the specific commit that added final_logit_softcapping to the converter
r2 = subprocess.run(
  'cd /workspace/llama.cpp && git log --oneline -S "final_logit_softcapping" -- convert_hf_to_gguf.py',
  shell=True, capture_output=True, text=True
)
print("softcap added to converter:", r2.stdout)


048a490f7 convert :  Mistral format yarn apply_scale support (#22612)
db44417b0 convert : apply Q/K RoPE permutation in NVFP4 repack path (#22611)
63d93d173 convert : disable uint types (#18908)
6118c043b ci : bump ty to 0.0.33 (#22535)
5d56effde convert : add support for Nemotron Nano 3 Omni (#22481)
d13540bec convert : remove input_scale for dequantized fp8 modelopt (#22356)
5eaee6538 convert : Handle ModelOpt produced mixed precision model during convert to GGUF (#22247)
7bfe60fdf mtmd, llama : Update HunyuanVL vision-language model support (#22037)
8685e7b07 convert : support sentence-transformer 5.4 config files (#22087)
89a5474f0 convert : fix (ignore for now) typings errors (#22002)
03b3d0779 Convert: Fix NemotronH Config Parsing (#21664)
21a493304 mtmd: qwen3 audio support (qwen3-omni and qwen3-asr) (#19441)
1e9d771e2 convert : force f16 or f32 on step3-vl conv weights (#21646)
073bb2c20 mtmd : add MERaLiON-2 multimodal audio support (#21756)
c8ac02fa1 requirements : update tra

In [10]:
import sys                                                                                                                                                                                                                        
sys.path.insert(0, '/workspace/llama.cpp/gguf-py')
from gguf import GGUFReader

reader = GGUFReader('/workspace/aria-v2-q8.gguf', 'r')
for name in sorted(reader.fields.keys()):
  print(name)


GGUF.kv_count
GGUF.tensor_count
GGUF.version
gemma4.attention.head_count
gemma4.attention.head_count_kv
gemma4.attention.key_length
gemma4.attention.key_length_swa
gemma4.attention.layer_norm_rms_epsilon
gemma4.attention.shared_kv_layers
gemma4.attention.sliding_window
gemma4.attention.sliding_window_pattern
gemma4.attention.value_length
gemma4.attention.value_length_swa
gemma4.block_count
gemma4.context_length
gemma4.embedding_length
gemma4.embedding_length_per_layer_input
gemma4.feed_forward_length
gemma4.final_logit_softcapping
gemma4.rope.dimension_count
gemma4.rope.dimension_count_swa
gemma4.rope.freq_base
gemma4.rope.freq_base_swa
general.architecture
general.file_type
general.name
general.quantization_version
general.size_label
general.type
tokenizer.chat_template
tokenizer.ggml.add_bos_token
tokenizer.ggml.add_space_prefix
tokenizer.ggml.bos_token_id
tokenizer.ggml.eos_token_id
tokenizer.ggml.mask_token_id
tokenizer.ggml.merges
tokenizer.ggml.model
tokenizer.ggml.padding_token_

In [12]:
import subprocess, time

minimal = 'FROM /workspace/aria-v2-q8.gguf\nPARAMETER num_ctx 512\n'
with open('/tmp/Modelfile.minimal', 'w') as f:
  f.write(minimal)

subprocess.run(['ollama', 'create', 'aria-minimal', '-f', '/tmp/Modelfile.minimal'],
             capture_output=True, text=True)

# Start Ollama if not running
subprocess.Popen(['ollama', 'serve'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(4)

import urllib.request, json
payload = json.dumps({'model': 'aria-minimal',
                    'messages': [{'role': 'user', 'content': 'What is your name?'}],
                    'stream': False}).encode()
req = urllib.request.Request('http://localhost:11434/api/chat', data=payload,
                           headers={'Content-Type': 'application/json'})
with urllib.request.urlopen(req, timeout=60) as resp:
  print(json.loads(resp.read())['message']['content'][:300])


<unused27><unused26><mask><unused24><unused12><unused30><unused27><mask><unused17><unused8><unused30><unused31><unused2><unused15><unused5>[multimodal]<unused31><unused13><unused11><unused1><unused25><unused25><unused20><unused8><unused9><unused22><unused19><unused6><unused18><unused20><unused0><unu


In [13]:
'''
That's the exact commit that broke Ollama compatibility. 63f8fe0ef renamed Gemma 4 from gemma3 → gemma4 in the converter. Ollama 0.23.0's bundled llama.cpp only knows gemma3.                                                    
                                                                                                                                                                                                                                    
  The good news: convert_hf_to_gguf.py is a Python script — no rebuild needed. Just check out the parent commit and the script is immediately usable.                                                                               
'''
import subprocess                                                                                                                                                                                                                 
              
# Checkout the commit just BEFORE gemma4 was introduced
r = subprocess.run(
  'cd /workspace/llama.cpp && git checkout 63f8fe0ef~1',
  shell=True, capture_output=True, text=True
)
print(r.stderr[-200:])

# Verify gemma4 is gone from the converter
r2 = subprocess.run(
  'grep -c "gemma4" /workspace/llama.cpp/convert_hf_to_gguf.py || echo 0',
  shell=True, capture_output=True, text=True
)
print("gemma4 occurrences:", r2.stdout.strip())

# Record the commit hash for Cell 8a pinning
r3 = subprocess.run(
  'cd /workspace/llama.cpp && git rev-parse --short HEAD',
  shell=True, capture_output=True, text=True
)
print("Pinned commit:", r3.stdout.strip())


ndo this operation with:

  git switch -

Turn off this advice by setting config variable advice.detachedHead to false

HEAD is now at 223373742 common : add commentary rules for gpt-oss-20b (#21286)

gemma4 occurrences: 0
0
Pinned commit: 223373742


## Cell 15 -- Connect Local Dash App

On your **local Windows machine**:

```powershell
$env:OLLAMA_BASE_URL = "http://<vast-ai-ip>:11434/v1"
$env:OLLAMA_MODEL    = "aria-v2"
.venv\Scripts\python.exe application.py
```

Or use a Cloudflare tunnel on the vast.ai instance:
```bash
cloudflared tunnel --url http://localhost:11434
```
